# 05. PFN 논문 재현

`04_pfn_gp_reproduction.ipynb`의 학습을 GPU 에서 돌리는 스크립트 입니다. 

job parameter로 규모를 바꿀 수 있습니다:
- `config_name` — `small` / `large` / `paper` (기본 `large`)
- `steps` — config의 기본 step 수를 덮어씀 (비우면 config 기본값). **처음엔 짧게(예: 500) 실행 확인**, 그다음 늘려 재현.

In [ ]:
try:
    import pfn_cookbook
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import pfn_cookbook

print(pfn_cookbook.__file__)

In [ ]:
import os
from dataclasses import replace

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch

from pfn_cookbook import (
    PFNConfig,
    analytic_gp_posterior,
    build_criterion,
    build_model,
    build_prior_sampler,
    get_device,
    pfn_predict,
    set_seed,
    train_pfn,
)


def get_param(name, default):
    # Databricks job/notebook 파라미터 → 없으면 환경변수(PFN_<NAME>) → 기본값
    try:
        return dbutils.widgets.get(name)  # noqa: F821
    except Exception:
        return os.environ.get("PFN_" + name.upper(), default)


device = get_device()
if device.type == "mps":  # 로컬 MPS는 일부 연산 미지원 → CPU
    device = torch.device("cpu")
print("device:", device)
if device.type == "cuda":
    print("cuda  :", torch.version.cuda)
    print("gpu   :", torch.cuda.get_device_name(0))

In [ ]:
config_name = get_param("config_name", "large")
CONFIG = getattr(PFNConfig, config_name)()
steps_override = get_param("steps", "")
if steps_override:
    CONFIG = replace(CONFIG, steps=int(steps_override))
print("config_name:", config_name)
print(CONFIG)

## 학습

MLflow run을 열고 prior-fitting 루프를 돈다. `train_pfn(mlflow_log=True)`가 config를 params로, step별 train NLL과 `steps//10`마다 재는 held-out 검증 NLL(`val_nll`)을 metric으로 기록하고, 학습이 끝나면 **best(검증 최저)·final 모델을 `mlflow.pytorch`로 로깅**한다. 이미 열린 run에 기록되므로 아래 §검증 지표도 같은 run에 함께 남는다.

In [ ]:
set_seed(CONFIG.seed)
mlflow.end_run()  # 남아있는 run 정리 (재실행 안전)
run = mlflow.start_run(run_name=f"{config_name}-{CONFIG.steps}steps")
print("MLflow run_id:", run.info.run_id)

model = build_model(CONFIG).to(device)
criterion = build_criterion(CONFIG).to(device)
prior_sampler = build_prior_sampler(CONFIG)
print("parameters:", f"{model.num_parameters():,}")

history = train_pfn(
    model, prior_sampler, criterion, CONFIG, device=device, log_every=200, mlflow_log=True
)

losses = np.array(history["losses"])
fig_loss, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(losses, color="C0", alpha=0.4, lw=0.8)
if len(losses) >= 50:
    smooth = np.convolve(losses, np.ones(50) / 50, mode="valid")
    ax.plot(np.arange(len(smooth)) + 25, smooth, color="C0", lw=2)
ax.set_xlabel("step")
ax.set_ylabel("query NLL")
ax.set_title("Prior-fitting loss")
plt.show()
print("final ~", round(float(losses[-50:].mean()), 3))

## 검증 + MLflow 기록

노트북 03 §8과 동일한 setup으로 PFN 예측을 analytic GP posterior에 겹치고, RMSE·NLL을 MLflow metric으로, 체크포인트와 오버레이 그림을 MLflow 아티팩트로 남깁니다. 재현 확인/후속 로컬 로드는 이 run의 아티팩트에서 받습니다.

In [ ]:
X_grid = np.linspace(-5, 5, 200)
X_tr = np.array([-4.0, -2.5, -0.5, 1.0, 3.5])
set_seed(0)
y_tr = np.sin(X_tr) + 1e-4 * np.random.randn(X_tr.size)

mu, cov = analytic_gp_posterior(X_tr, y_tr, X_grid)
sd = np.sqrt(np.clip(np.diag(cov), 0, None))

pred = pfn_predict(
    model,
    criterion,
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32),
    torch.tensor(X_grid, dtype=torch.float32),
)
pfn_mean = pred["mean"].cpu().numpy()
pfn_sd = pred["std"].cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.fill_between(X_grid, mu - 2 * sd, mu + 2 * sd, color="C0", alpha=0.15, label="analytic ±2σ")
ax.plot(X_grid, mu, color="C0", lw=2, label="analytic mean")
ax.plot(X_grid, pfn_mean, color="C3", lw=2, ls="--", label="PFN mean")
ax.fill_between(X_grid, pfn_mean - 2 * pfn_sd, pfn_mean + 2 * pfn_sd, color="C3", alpha=0.15,
                label="PFN ±2σ")
ax.scatter(X_tr, y_tr, c="black", s=35, zorder=5, label="context D")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"PFN ({config_name}, {CONFIG.steps} steps) vs analytic GP posterior")
ax.legend(loc="upper left", fontsize=8)
plt.show()

rmse_mean = float(np.sqrt(np.mean((pfn_mean - mu) ** 2)))
mean_abs_dsigma = float(np.mean(np.abs(pfn_sd - sd)))

Xq = np.linspace(-4.5, 4.5, 40)
yq = np.sin(Xq)
mu_q, cov_q = analytic_gp_posterior(X_tr, y_tr, Xq)
var_q = np.clip(np.diag(cov_q), 1e-9, None) + 1e-4**2
gp_nll = float(np.mean(0.5 * np.log(2 * np.pi * var_q) + 0.5 * (yq - mu_q) ** 2 / var_q))
predq = pfn_predict(
    model,
    criterion,
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32),
    torch.tensor(Xq, dtype=torch.float32),
)
pfn_nll = float(criterion(predq["logits"], torch.tensor(yq, dtype=torch.float32)).mean())

for k, v in {
    "rmse_mean": rmse_mean,
    "mean_abs_dsigma": mean_abs_dsigma,
    "pfn_nll": pfn_nll,
    "gp_nll": gp_nll,
    "nll_gap": pfn_nll - gp_nll,
}.items():
    mlflow.log_metric(k, v)

fig.savefig("/tmp/pfn_overlay.png", dpi=120, bbox_inches="tight")
mlflow.log_artifact("/tmp/pfn_overlay.png")
mlflow.end_run()

print(f"RMSE(mean)={rmse_mean:.4f}  mean|Δσ|={mean_abs_dsigma:.4f}")
print(f"PFN NLL={pfn_nll:.3f}  GP NLL(하한)={gp_nll:.3f}  gap={pfn_nll - gp_nll:+.3f}")
print("logged overlay + best/final models to MLflow run", run.info.run_id)